## Deep Neural Network 
Build a deep neural network to classify MNIST digits

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [3]:
# Load MNIST and create train/test split

# Convert images to tensors and normalize with mean and std of MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
]) 

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Number of samples model processes at once during training/testing 
# Model sees 64 images at a time, computes loss and back propagates gradients, then SGD updates weights. 
batch_size = 64 

# Create data loaders to handle batching and shuffling of data during training/testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 60000
Test size: 10000


In [5]:
model = nn.Sequential(
    nn.Flatten(),              # Convert 28x28 image tensors into 784-length 1D vectors, [Batch, 1, 28, 28] -> [Batch, 784]
    nn.Linear(784, 64),        # Hidden layer 1 with 64 neurons, batch normalization and ReLU activation 
    nn.BatchNorm1d(64),
    nn.ReLU(),

                        
    nn.Linear(64, 64),         # Hidden layer 2 with 64 neurons
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 64),         # Hidden layer 3 with 64 neurons
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.2),           # dropout rate to prevent overfitting

    nn.Linear(64, 10),         # Output layer for 10 MNIST classes
    #nn.Softmax(dim=1)         # alrealdy included in the loss function
)

In [6]:
print(model)
total_params = sum(p.numel() for p in model.parameters())
print("Total model parameters:", total_params)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=64, bias=True)
  (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=64, bias=True)
  (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=64, bias=True)
  (8): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (9): ReLU()
  (10): Dropout(p=0.2, inplace=False)
  (11): Linear(in_features=64, out_features=10, bias=True)
)
Total model parameters: 59594


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) # Adam optimizer 

In [8]:
# Model training loop with GPU support if available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

epochs = 30
for epoch in range(epochs):
    model.train() # Set model to training mode (enables dropout, batch norm, etc. if present)
    running_loss = 0.0 
    correct = 0
    total = 0

    # Iterate over mini-batches from the training loader
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        targets = nn.functional.one_hot(labels, num_classes=10).float() # Convert labels to one-hot encoding 

        optimizer.zero_grad()
        outputs = model(images) # Forward pass: compute predicted probabilities for the current batch
        loss = criterion(outputs, targets) # Compute loss between predicted probabilities and one-hot encoded true labels
        loss.backward() # Backward pass: compute gradients of the loss with respect to model parameters
        optimizer.step() # Update model parameters based on computed gradients and learning rate

        # Track loss and accuracy for the current epoch
        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1) # Get predicted class 
        correct += (predicted == labels).sum().item() # Count correct predictions in the current batch
        total += labels.size(0)

    train_loss = running_loss / total # Epoch average loss per sample
    train_acc = correct / total # Epoch training accuracy 

    # Evaluate on the test split without computing gradients
    model.eval() # Set model to evaluation mode (disables dropout, batch norm updates, etc.)
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images) # Forward pass on test data to compute predicted probabilities
            predicted = outputs.argmax(dim=1) # Get predicted class
            test_correct += (predicted == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total
    print(f"Epoch {epoch + 1}/{epochs} - loss: {train_loss:.4f} - train acc: {train_acc:.4f} - test acc: {test_acc:.4f}")

Using device: mps
Epoch 1/30 - loss: 0.2633 - train acc: 0.9216 - test acc: 0.9619
Epoch 2/30 - loss: 0.1407 - train acc: 0.9588 - test acc: 0.9687
Epoch 3/30 - loss: 0.1143 - train acc: 0.9657 - test acc: 0.9755
Epoch 4/30 - loss: 0.0997 - train acc: 0.9693 - test acc: 0.9733
Epoch 5/30 - loss: 0.0876 - train acc: 0.9728 - test acc: 0.9749
Epoch 6/30 - loss: 0.0792 - train acc: 0.9762 - test acc: 0.9765
Epoch 7/30 - loss: 0.0723 - train acc: 0.9771 - test acc: 0.9783
Epoch 8/30 - loss: 0.0676 - train acc: 0.9793 - test acc: 0.9794
Epoch 9/30 - loss: 0.0655 - train acc: 0.9799 - test acc: 0.9795
Epoch 10/30 - loss: 0.0594 - train acc: 0.9815 - test acc: 0.9778
Epoch 11/30 - loss: 0.0566 - train acc: 0.9815 - test acc: 0.9783
Epoch 12/30 - loss: 0.0510 - train acc: 0.9838 - test acc: 0.9812
Epoch 13/30 - loss: 0.0493 - train acc: 0.9843 - test acc: 0.9795
Epoch 14/30 - loss: 0.0465 - train acc: 0.9848 - test acc: 0.9785
Epoch 15/30 - loss: 0.0459 - train acc: 0.9855 - test acc: 0.9800
E